In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import matplotlib.pyplot as plt
import seaborn as sns

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
train_data = pd.read_csv("/kaggle/input/titanic/train.csv")
train_data.head()

In [ ]:
train_data.info()

In [ ]:
train_data.describe()

In [ ]:
train_data.columns

In [ ]:
train_data.isnull().sum()

## EDA

In [ ]:
train_data.columns

In [ ]:
train_data['PassengerId'].nunique()

In [ ]:

## Checking a range of age
bins = [0,5,18,30,50,60,100]
labels=['0-5', '6-18', '19-30', '31-50', '51-60',  '61+']

train_data['Age_range'] = pd.cut(train_data['Age'], bins=bins, labels=labels)

print(train_data['Age_range'].value_counts().sort_index())

#### Gender - Age Insights

In [ ]:
train_data['Sex'].value_counts()

In [ ]:
## Out of all the men, how many survived?
train_data.iloc[np.where(train_data['Sex']=='male')][['Sex', 'Survived']].value_counts()

In [ ]:
## Out of all women, how many survived?
train_data.iloc[np.where(train_data['Sex']=='female')][['Sex', 'Survived']].value_counts()

In [ ]:
survival_group = train_data.groupby(['Age_range', 'Sex', 'Survived']).size().reset_index(name='Count')
print(survival_group)

In [ ]:
g = sns.catplot(
    data=train_data,
    x='Age_range',
    hue='Survived',
    col='Sex',
    kind='count',
    palette={0: 'salmon', 1: 'steelblue'},
    height=5,
    aspect=1.2
)

g.set_axis_labels("Age Range", "Count")
g.set_titles("{col_name}")
g.add_legend(title='Survived', labels=['No', 'Yes'])
plt.suptitle('Survival by Age Group and Gender', y=1.02, fontweight='bold')
plt.tight_layout()
plt.show()


#### Observation on Survival by Age and Gender
- More women survived compared to men
- Men around the age of 19-50 did not survive
- Children around the age of 0-5 have survived

This makes us understand that age and gender are ciritcal features for our model

#### Bringing in Pclass

In [ ]:
train_data['Pclass'].value_counts().sort_index()

In [ ]:
train_data[
    (train_data['Sex'] == 'male') & (train_data['Pclass'] == 1)
][['Survived', 'Pclass', 'Sex']].value_counts()

In [ ]:
train_data[
    (train_data['Sex'] == 'male') & (train_data['Pclass'] == 2)
][['Survived', 'Pclass', 'Sex']].value_counts()

In [ ]:
train_data[
    (train_data['Sex'] == 'male') & (train_data['Pclass'] == 3)
][['Survived', 'Pclass', 'Sex']].value_counts()

In [ ]:
train_data[
    (train_data['Sex'] == 'female') & (train_data['Pclass'] == 1)
][['Survived', 'Pclass', 'Sex']].value_counts()

In [ ]:
train_data[
    (train_data['Sex'] == 'female') & (train_data['Pclass'] == 2)
][['Survived', 'Pclass', 'Sex']].value_counts()

In [ ]:
train_data[
    (train_data['Sex'] == 'female') & (train_data['Pclass'] == 3)
][['Survived', 'Pclass', 'Sex']].value_counts()

In [ ]:
age_order = ['0-5', '6-15', '16-30', '31-50', '51-70', '71-80', '81+']

g = sns.catplot(
    data=train_data,
    x='Age_range',
    hue='Survived',
    col='Sex',
    row='Pclass',
    kind='count',
    order=age_order,  # ← forces correct order on x-axis
    palette={0: 'salmon', 1: 'steelblue'},
    height=4,
    aspect=1.2
)

g.set_axis_labels("Age Range", "Count")
g.set_titles("{row_name} class | {col_name}")
g.add_legend(title='Survived', labels=['No', 'Yes'])

# ── Rotate x-axis labels so they don't overlap ──────────────────
for ax in g.axes.flat:
    ax.tick_params(axis='x', rotation=45)

plt.suptitle('Survival by Pclass, Age Group and Gender', y=1.02, fontweight='bold')
plt.tight_layout()
plt.show()

#### Age, Gender, Pclass
- More men died in thrid class
- More have survived in first class.
- Passenger Class played an important role here

In [ ]:
train_data.columns

#### Sibsp/PArch

In [ ]:
train_data['SibSp'].value_counts()

In [ ]:
train_data['Parch'].value_counts()

In [ ]:
train_data.iloc[np.where(train_data['SibSp']==0)][['SibSp', 'Survived']].value_counts()

In [ ]:
train_data.iloc[np.where(train_data['SibSp']==1)][['SibSp', 'Survived']].value_counts()

In [ ]:
train_data.iloc[np.where(train_data['Parch']==0)][['Parch', 'Survived']].value_counts()

In [ ]:
train_data.iloc[np.where(train_data['Parch']!=0)][['Parch', 'Survived']].value_counts()

In [ ]:
train_data['Family_Size'] = train_data['SibSp'] + train_data['Parch'] + 1

In [ ]:
sns.catplot(
    data=train_data,
    x='Family_Size',
    hue='Survived',
    kind='count',
    palette={0: 'salmon', 1: 'steelblue'},
    height=5,
    aspect=1.2
)
plt.title('Survival by Family Size')
plt.show()

#### Observations on PArch/Sibsp
- Passengers who travelled alone did not survive
- Passengers who came with big familes did not survive

#### Introduce New Feature Family Size

In [ ]:
def family_category(size):
    if size == 1:
        return 'Alone'
    elif size<=3:
        return 'Small'
    elif size <=6:
        return 'Medium'
    else:
        return 'Large'

train_data['Family_Category'] = train_data['Family_Size'].apply(family_category)

In [ ]:
train_data['Family_Category'].value_counts()

#### Embarked/Fare/Pclass

In [ ]:
train_data['Embarked'].value_counts()

In [ ]:
train_data[
    (train_data['Embarked'] == 'S') & (train_data['Pclass'] == 1)
][['Embarked', 'Pclass', 'Survived']].value_counts()

In [ ]:
train_data[
    (train_data['Embarked'] == 'S') & (train_data['Pclass'] == 2)
][['Embarked', 'Pclass', 'Survived']].value_counts()

In [ ]:
train_data[
    (train_data['Embarked'] == 'S') & (train_data['Pclass'] == 3)
][['Embarked', 'Pclass', 'Survived']].value_counts()

In [ ]:
train_data[
    (train_data['Embarked'] == 'C') & (train_data['Pclass'] == 1)
][['Embarked', 'Pclass', 'Survived']].value_counts()

In [ ]:
train_data[
    (train_data['Embarked'] == 'C') & (train_data['Pclass'] == 2)
][['Embarked', 'Pclass', 'Survived']].value_counts()

In [ ]:
train_data[
    (train_data['Embarked'] == 'C') & (train_data['Pclass'] == 3)
][['Embarked', 'Pclass', 'Survived']].value_counts()

In [ ]:
train_data[
    (train_data['Embarked'] == 'Q') & (train_data['Pclass'] == 1)
][['Embarked', 'Pclass', 'Survived']].value_counts()

In [ ]:
train_data[
    (train_data['Embarked'] == 'Q') & (train_data['Pclass'] == 2)
][['Embarked', 'Pclass', 'Survived']].value_counts()

In [ ]:
train_data[
    (train_data['Embarked'] == 'Q') & (train_data['Pclass'] == 3)
][['Embarked', 'Pclass', 'Survived']].value_counts()

In [ ]:

fare_bins = [0, 10, 30, 60, 100, 600]
fare_labels = ['0-10', '11-30', '31-60', '61-100', '100+']

train_data['Fare_Range'] = pd.cut(train_data['Fare'], bins=fare_bins, labels=fare_labels)

sns.catplot(
    data=train_data,
    x='Fare_Range',
    hue='Survived',
    kind='count',
    palette={0: 'salmon', 1: 'steelblue'},
    height=5,
    aspect=1.2
)
plt.title('Survival by Fare Range')
plt.show()

sns.catplot(
    data=train_data,
    x='Embarked',
    hue='Survived',
    kind='count',
    palette={0: 'salmon', 1: 'steelblue'},
    height=5,
    aspect=1.2
)
plt.title('Survival by Embarked Port')
plt.show()

#### Observations
- More people boarded from Southampton where more from 3rd class have not survived
- Very less first class people boarded from Queenstown

# Feature Engineering

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

corr_data = train_data.copy()
corr_data['Sex'] = corr_data['Sex'].map({'male': 0, 'female': 1})
corr_data['Embarked'] = corr_data['Embarked'].map({'S': 0, 'C': 1, 'Q': 2})

features = ['Survived', 'Pclass', 'Sex', 'Age', 'SibSp', 
            'Parch', 'Fare', 'Embarked', 'Family_Size']

corr_matrix = corr_data[features].corr()


plt.figure(figsize=(10, 8))
sns.heatmap(
    corr_matrix,
    annot=True,        # show values
    fmt='.2f',         # 2 decimal places
    cmap='coolwarm',   # red = positive, blue = negative
    center=0,
    square=True,
    linewidths=0.5
)
plt.title('Feature Correlation Heatmap', fontweight='bold')
plt.tight_layout()
plt.show()

#### Observations
- This heatmap shows that SibSp and Parch are hihgly correlated with family size. So it's best we drop the redundant columns
- Sex is highly correlated with the target since it's likely that more women survived

#### Dropping columns

In [ ]:
train_data.columns

In [ ]:
train_data = train_data.drop(columns=['Age_range', 'Fare_Range', 'PassengerId', 'Name','Ticket', 'SibSp', 'Parch', 'Name', 'PassengerId', 'Cabin'])

In [ ]:
train_data.columns

In [ ]:
train_data['Pclass'].isna().sum()

In [ ]:
train_data['Sex'].isna().sum()

In [ ]:
train_data['Embarked'].isna().sum()

In [ ]:
train_data['Family_Category'].isna().sum()

In [ ]:
train_data.info()

In [ ]:
train_data['Sex'] = train_data['Sex'].map({'male': 0, 'female': 1})

In [ ]:
train_data = pd.get_dummies(train_data, columns=['Embarked', 'Family_Category'], drop_first=True)


In [ ]:
train_data.info()

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(12, 10))
sns.heatmap(
    train_data.corr(),
    annot=True,
    fmt='.2f',
    cmap='coolwarm',
    center=0,
    square=True,
    linewidths=0.5
)
plt.title('Correlation Heatmap After Encoding', fontweight='bold')
plt.tight_layout()
plt.show()



vif_data = pd.DataFrame()
vif_data['Feature'] = features.columns
vif_data['VIF'] = [variance_inflation_factor(features.values, i) 
                   for i in range(len(features.columns))]

print(vif_data.sort_values('VIF', ascending=False))

In [ ]:
train_data.drop(columns=['Family_Size'], inplace=True)

#### VIF

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor
import pandas as pd


features = train_data.drop(columns=['Survived']).dropna()

# ── Check what type it is ────────────────────────────────────────
print(type(features))

vif_data = pd.DataFrame()
vif_data['Feature'] = features.columns
vif_data['VIF'] = [variance_inflation_factor(features.values, i) 
                   for i in range(len(features.columns))]

print(vif_data.sort_values('VIF', ascending=False))

In [ ]:
train_data['Age'] = train_data.groupby(['Pclass', 'Sex'])['Age'].transform(
    lambda x: x.fillna(x.median())
)

# Model Training

#### Logistic Regression

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

In [ ]:
y = train_data['Survived']

In [ ]:
X = train_data.drop(columns=['Survived'])

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train, y_train)

In [ ]:
y_pred = lr.predict(X_val)

In [ ]:
print(f"\nAccuracy : {accuracy_score(y_val, y_pred):.4f}")
print("\nClassification Report:")
print(classification_report(y_val, y_pred))

In [ ]:
sns.heatmap(
    confusion_matrix(y_val, y_pred),
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=['Not Survived', 'Survived'],
    yticklabels=['Not Survived', 'Survived']
)

#### Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, roc_curve, auc)
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

In [ ]:
y_pred_rf = rf.predict(X_val)
y_prob_rf  = rf.predict_proba(X_val)[:, 1]

In [ ]:
print(f"Accuracy : {accuracy_score(y_val, y_pred_rf):.4f}")
print("\nClassification Report:")
print(classification_report(y_val, y_pred_rf))

In [ ]:
plt.figure(figsize=(6, 4))
sns.heatmap(
    confusion_matrix(y_val, y_pred_rf),
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=['Not Survived', 'Survived'],
    yticklabels=['Not Survived', 'Survived']
)
plt.title('Random Forest — Confusion Matrix')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.show()

#### XGBoost

In [ ]:
from xgboost import XGBClassifier

In [ ]:
xgb = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=4,
    reg_alpha=0,
    reg_lambda=1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

In [ ]:
xgb.fit(X_train, y_train,
       eval_set=[(X_val,y_val)],
       early_stopping_rounds=20,
       verbose=False)

In [ ]:
y_pred_xg = xgb.predict(X_val)
y_prob_xg  = xgb.predict_proba(X_val)[:, 1]

In [ ]:
print(f"Accuracy : {accuracy_score(y_val, y_pred_xg):.4f}")
print("\nClassification Report:")
print(classification_report(y_val, y_pred_xg))

In [ ]:
plt.figure(figsize=(6, 4))
sns.heatmap(
    confusion_matrix(y_val, y_pred_xg),
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=['Not Survived', 'Survived'],
    yticklabels=['Not Survived', 'Survived']
)
plt.title('XGBoost — Confusion Matrix')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.show()

#### 3. Gradient Boosting

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier


In [ ]:
gb = GradientBoostingClassifier(n_estimators=100, random_state=42)
gb.fit(X_train,y_train)

In [ ]:
y_pred_gb = gb.predict(X_val)
y_prob_gb  = gb.predict_proba(X_val)[:, 1]

In [ ]:
print(f"Accuracy : {accuracy_score(y_val, y_pred_gb):.4f}")

In [ ]:
print(classification_report(y_val, y_pred_gb))

In [ ]:
plt.figure(figsize=(6, 4))
sns.heatmap(
    confusion_matrix(y_val, y_pred_gb),
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=['Not Survived', 'Survived'],
    yticklabels=['Not Survived', 'Survived']
)
plt.title('Gradient Boosting — Confusion Matrix')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.show()

#### K Nearest Neighbors

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled   = scaler.transform(X_val)

In [ ]:
k_scores = []
k_range  = range(1, 21)

In [ ]:
for k in k_range:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train_scaled, y_train)
    k_scores.append(accuracy_score(y_val, knn.predict(X_val_scaled)))

In [ ]:
knn = KNeighborsClassifier(n_neighbors=best_k)
knn.fit(X_train_scaled, y_train)

In [ ]:
y_pred_knn = knn.predict(X_val_scaled)
y_prob_knn  = knn.predict_proba(X_val_scaled)[:, 1]

In [ ]:
print(f"\nAccuracy : {accuracy_score(y_val, y_pred_knn):.4f}")

In [ ]:
print("\nClassification Report:")
print(classification_report(y_val, y_pred_knn))

In [ ]:
plt.figure(figsize=(6, 4))
sns.heatmap(
    confusion_matrix(y_val, y_pred_knn),
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=['Not Survived', 'Survived'],
    yticklabels=['Not Survived', 'Survived']
)
plt.title(f'KNN (K={best_k}) — Confusion Matrix')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.show()

#### Neural Networks

In [ ]:
from sklearn.neural_network import MLPClassifier

In [ ]:
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled   = scaler.transform(X_val)

In [ ]:
nn = MLPClassifier(
    hidden_layer_sizes=(64, 24),   
    activation='relu',             
    solver='adam',                 
    learning_rate_init=0.05,     
    max_iter=450,                  
    random_state=42
)

In [ ]:
nn.fit(X_train_scaled, y_train)

In [ ]:
y_pred_nn = nn.predict(X_val_scaled)
y_prob_nn  = nn.predict_proba(X_val_scaled)[:, 1]

In [ ]:
print(f"Accuracy : {accuracy_score(y_val, y_pred_nn):.4f}")

# ── 5. Classification Report ─────────────────────────────────────
print("\nClassification Report:")
print(classification_report(y_val, y_pred_nn))

In [ ]:
plt.figure(figsize=(6, 4))
sns.heatmap(
    confusion_matrix(y_val, y_pred_nn),
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=['Not Survived', 'Survived'],
    yticklabels=['Not Survived', 'Survived']
)
plt.title('Neural Network — Confusion Matrix')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.metrics import roc_curve, auc
import matplotlib.pyplot as plt

# ── 1. Define all trained models and their probabilities ─────────
models_prob = {
    'Logistic Regression' : (lr,     X_val,        'blue'),
    'Random Forest'       : (rf,     X_val,        'steelblue'),
    'XGBoost'             : (xgb,    X_val,        'orange'),
    'Gradient Boosting'   : (gb,     X_val,        'darkorange'),
    'KNN'                 : (knn,    X_val_scaled, 'green'),
    'Neural Network'      : (nn,     X_val_scaled, 'purple'),
}

# ── 2. Plot all ROC curves together ──────────────────────────────
plt.figure(figsize=(10, 7))

for name, (model, X, color) in models_prob.items():
    y_prob      = model.predict_proba(X)[:, 1]
    fpr, tpr, _ = roc_curve(y_val, y_prob)
    roc_auc     = auc(fpr, tpr)
    plt.plot(fpr, tpr, color=color, label=f'{name} (AUC = {roc_auc:.3f})', linewidth=2)

# ── 3. Random chance baseline ────────────────────────────────────
plt.plot([0, 1], [0, 1], 'k--', label='Random Chance (AUC = 0.500)', linewidth=1.5)

# ── 4. Formatting ────────────────────────────────────────────────
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('ROC Curve — All Models Comparison', fontsize=14, fontweight='bold')
plt.legend(loc='lower right', fontsize=10)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# ── 5. AUC Summary Table ─────────────────────────────────────────
auc_summary = []

for name, (model, X, _) in models_prob.items():
    y_prob      = model.predict_proba(X)[:, 1]
    fpr, tpr, _ = roc_curve(y_val, y_prob)
    roc_auc     = auc(fpr, tpr)
    auc_summary.append({'Model': name, 'AUC': round(roc_auc, 4)})

auc_df = pd.DataFrame(auc_summary).sort_values('AUC', ascending=False)
print("\n=== AUC Summary ===")
print(auc_df.to_string(index=False))

# Test Data

In [ ]:
import pandas as pd


test_data = pd.read_csv("/kaggle/input/titanic/test.csv")

passenger_ids = test_data['PassengerId']

# ── 3. Apply same preprocessing as train data ────────────────────

# Drop unnecessary columns
test_data.drop(columns=['PassengerId', 'Name', 'Ticket', 'Cabin'], inplace=True)

# Encode Sex
test_data['Sex'] = test_data['Sex'].map({'male': 0, 'female': 1})

# Fill missing Embarked
test_data['Embarked'] = test_data['Embarked'].fillna('S')

# Engineer Family Size and Category
test_data['Family_Size'] = test_data['SibSp'] + test_data['Parch'] + 1

def family_category(size):
    if size == 1:
        return 'Alone'
    elif size <= 3:
        return 'Small'
    elif size <= 6:
        return 'Medium'
    else:
        return 'Large'

test_data['Family_Category'] = test_data['Family_Size'].apply(family_category)

# Drop SibSp, Parch, Family_Size
test_data.drop(columns=['SibSp', 'Parch', 'Family_Size'], inplace=True)

# One Hot Encode Embarked and Family_Category
test_data = pd.get_dummies(test_data, columns=['Embarked', 'Family_Category'], drop_first=True)

# Fill missing Age using group median
test_data['Age'] = test_data.groupby(['Pclass', 'Sex'])['Age'].transform(
    lambda x: x.fillna(x.median())
)

# Fill missing Fare if any
test_data['Fare'] = test_data['Fare'].fillna(test_data['Fare'].median())

# ── 4. Align test columns with train columns ─────────────────────
# Ensure test has same columns as training data
missing_cols = set(X_train.columns) - set(test_data.columns)
for col in missing_cols:
    test_data[col] = 0

# Reorder columns to match training
test_data = test_data[X_train.columns]

print("Test data shape  :", test_data.shape)
print("Train data shape :", X_train.shape)
print("\nTest columns match train:", list(test_data.columns) == list(X_train.columns))

# ── 5. Predict using best model (Random Forest) ──────────────────
predictions = rf.predict(test_data)

# ── 6. Create submission file ────────────────────────────────────
submission = pd.DataFrame({
    'PassengerId' : passenger_ids,
    'Survived'    : predictions
})

submission.to_csv('submission.csv', index=False)

print(submission.head(10))
print(f"\nTotal predictions : {len(submission)}")
print("Survived count:", submission['Survived'].sum())
print("Not Survived count:", (submission['Survived'] == 0).sum())